# T2.6 – DBRepo API Reimplementation

This notebook documents the DBRepo-based reimplementation of the experiment.

The final experiment pipeline retrieves data exclusively from DBRepo and does not read local CSV files during training or evaluation. The implementation is located in:

- `src/data/load_from_dbrepo.py`
- `src/features/build_features.py`
- `src/models/train_model.py`
- `src/models/evaluate_model.py`
- `src/pipeline/run_experiment.py`

The pipeline is executed with:

```bash
python -m src.pipeline.run_experiment

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()

if repo_root.name == "notebooks":
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("Repository root:", repo_root)

Repository root: /Users/kerimhalilovic/Documents/GitHub/Vienna-Weather-Wet-Month-Prediction



### Config check cell

In [8]:
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(), override=True)

print("DBRepo endpoint:", os.getenv("DBREPO_ENDPOINT"))
print("Database ID:", os.getenv("DBREPO_DATABASE_ID"))
print("Weather table ID:", os.getenv("DBREPO_TABLE_WEATHER_MEASUREMENT_ID"))
print("Time table ID:", os.getenv("DBREPO_TABLE_TIME_DIMENSION_ID"))
print("Station table ID:", os.getenv("DBREPO_TABLE_STATION_ID"))
print("Username configured:", os.getenv("DBREPO_USERNAME") is not None)
print("Password configured:", os.getenv("DBREPO_PASSWORD") is not None)

DBRepo endpoint: https://test.dbrepo.tuwien.ac.at
Database ID: a181cad5-4bdb-48b2-937e-3e75293f6a7b
Weather table ID: 3674fea3-a7be-4dfe-8356-bc692bd1ff6c
Time table ID: fa248a2c-bfb6-4d8e-a89b-2dbd19ab8cde
Station table ID: ab02386c-e27c-4c1f-a27d-93034ce3fa79
Username configured: True
Password configured: True


### Loader test cell

In [9]:
from src.data.load_from_dbrepo import load_weather_data_from_dbrepo

df = load_weather_data_from_dbrepo()

print("Rows loaded:", len(df))
print("Columns:", len(df.columns))
display(df.head())

/Users/kerimhalilovic/Documents/GitHub/Vienna-Weather-Wet-Month-Prediction/src/data/load_from_dbrepo.py:65: RuntimeWarning: DBRepo view 'weather_measurement_v2_features' is missing ML feature columns ['altitude_m', 'latitude_deg', 'longitude_deg', 'ref_month', 'ref_year']; falling back to base tables.
  view_df = _try_load_feature_view(client, database_id)


Rows loaded: 1845
Columns: 35


,mean_t_max_c,mean_t_min_c,measurement_id,num_clear,num_cloud,num_frost,num_heat,num_ice,num_precp_01,num_summer,...,wind_vel_max_ms,ref_month,ref_year,altitude_m,district_code,latitude_deg,longitude_deg,nuts_code,station_name,sub_district_code
0,18.6,9.3,697,4,8,0,0,0,15,1,...,NaN,5,1930,202.0,91900,48.248611,16.356944,AT13,Wien - Hohe Warte,91905
1,9.6,3.8,823,3,12,1,0,0,9,0,...,NaN,11,1940,202.0,91900,48.248611,16.356944,AT13,Wien - Hohe Warte,91905
2,11.3,3.2,1774,1,12,3,0,0,15,0,...,108.0,2,2020,202.0,91900,48.248611,16.356944,AT13,Wien - Hohe Warte,91905
3,19.5,11.4,1277,2,12,0,0,0,16,3,...,100.0,9,1978,202.0,91900,48.248611,16.356944,AT13,Wien - Hohe Warte,91905
4,11.8,0.5,971,14,4,11,0,0,7,0,...,83.0,3,1953,202.0,91900,48.248611,16.356944,AT13,Wien - Hohe Warte,91905


### Full experiment cell

In [11]:
import subprocess
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

result = subprocess.run(
    [sys.executable, "-m", "src.pipeline.run_experiment"],
    cwd=repo_root,
    text=True,
    capture_output=True
)

print(result.stdout)
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(f"Experiment failed with return code {result.returncode}")

DBRepo wet-month experiment complete.
Rows loaded: 1845
Features used: 27
Split strategy: chronological

Test metrics:
       model  accuracy  precision  recall    f1
      logreg     0.747      0.688   0.440 0.537
randomforest     0.760      0.706   0.480 0.571

Output artefacts:
- confusion_matrix_logreg: outputs/figures/fig_confusion_matrix_logreg_v1.png
- confusion_matrix_randomforest: outputs/figures/fig_confusion_matrix_randomforest_v1.png
- metrics: outputs/predictions/model_metrics_v1.csv
- model_comparison: outputs/figures/fig_model_comparison_v1.png
- model_logreg: outputs/models/model_logreg_v1.pkl
- model_randomforest: outputs/models/model_randomforest_v1.pkl
- predictions: outputs/predictions/predictions_test_v1.csv

/Users/kerimhalilovic/Documents/GitHub/Vienna-Weather-Wet-Month-Prediction/src/data/load_from_dbrepo.py:65: RuntimeWarning: DBRepo view 'weather_measurement_v2_features' is missing ML feature columns ['altitude_m', 'latitude_deg', 'longitude_deg', 'ref_month',

### Metrics

In [12]:
import pandas as pd

metrics = pd.read_csv("../outputs/predictions/model_metrics_v1.csv")
display(metrics)

,model,accuracy,precision,recall,f1
0,logreg,0.746667,0.687500,0.44,0.536585
1,randomforest,0.760000,0.705882,0.48,0.571429


## Verification

The DBRepo-based experiment successfully loads 1845 cleaned observations from DBRepo, trains Logistic Regression and Random Forest classifiers, evaluates them using a chronological split, and writes model, prediction, metric, and figure artefacts to `outputs/`.

The final pipeline uses DBRepo API access only. Local CSV files are not used during model training or evaluation.